# Gephi GEXF -> gephi_node_metrics.csv

Gephi's **Export table** silently writes empty files when filters or the "Visible graph only"
setting interfere. **Export -> Graph file (GEXF)** uses a completely different code path and
carries every node attribute, so it sidesteps the problem.

### In Gephi
1. `File -> Export -> Graph file...`
2. Filename: **`network.gexf`**, save into `SNA_project/data/`
3. In the export options dialog, make sure **"Visible graph only" is UNCHECKED**
4. Export

Then run this notebook. It converts the GEXF into the exact `data/gephi_node_metrics.csv`
that `section1_sna_analysis.ipynb` expects — using **Gephi's own computed values**, so you can
legitimately write "computed in Gephi" in your methodology.

In [2]:
import networkx as nx, pandas as pd, numpy as np
from pathlib import Path

H = nx.read_gexf("data/network.gexf")
print(f"loaded: {H.number_of_nodes():,} nodes | {H.number_of_edges():,} edges")
print(f"directed: {H.is_directed()}")

if H.number_of_nodes() != 6434:
    print("\n*** WARNING: expected 6,434 nodes ***")
    print("A filter was probably still active, or 'Visible graph only' was checked.")
    print("Clear filters in Gephi and re-export.")
else:
    print("node count matches the validated dataset")

ParseError: no element found: line 1, column 0 (<string>)

In [ ]:
# inspect what Gephi actually attached to the nodes
sample = next(iter(H.nodes(data=True)))[1]
print("attributes found on nodes:")
for k, v in sample.items():
    print(f"  {k:<28} e.g. {v}")

In [ ]:
rows = []
for nid, attrs in H.nodes(data=True):
    r = {"Id": nid}
    r.update(attrs)
    rows.append(r)
df = pd.DataFrame(rows)

# GEXF stores a lowercase 'label' duplicate - drop it if a proper Label exists
if "label" in df.columns and "Label" in df.columns:
    df = df.drop(columns=["label"])
elif "label" in df.columns:
    df = df.rename(columns={"label": "Label"})

# Gephi sometimes writes numerics as strings
for col in df.columns:
    if col in ("Id", "Label"): continue
    df[col] = pd.to_numeric(df[col], errors="ignore")

print(f"{len(df):,} rows x {len(df.columns)} columns")
print("columns:", list(df.columns))
display(df.head())

In [ ]:
# report which metrics are present vs still missing
wanted = ["Modularity Class", "PageRank", "Betweenness Centrality",
          "Closeness Centrality", "Eigenvector Centrality"]
print("METRIC AVAILABILITY")
for w in wanted:
    hit = [c for c in df.columns if c.lower().replace("_"," ") == w.lower()]
    print(f"  {w:<26} {'FOUND: '+hit[0] if hit else 'MISSING - run this Statistic in Gephi'}")

df.to_csv("data/gephi_node_metrics.csv", index=False)
print(f"\nwrote data/gephi_node_metrics.csv ({len(df):,} rows)")
print(">>> now re-run the Block C and Block D cells in section1_sna_analysis.ipynb")

### If metrics are missing

`Modularity Class` only appears after running **Statistics -> Modularity**.
`Betweenness`/`Closeness` only appear after **Statistics -> Network Diameter** (10-30 min).
Run whichever are missing, re-export the GEXF, and re-run this notebook.

If the GEXF route also fails, use `section1b_local_metrics.ipynb` - it computes everything
in Python in ~25 seconds and writes the identical file. Your analysis is not blocked either way.